# Efficiency analysis

A numeric counterpart to `outcome_analysis.ipynb`: instead of classifying each round into one of three categorical outcomes, this scores each round by **efficiency** -- how close the pair came to the best possible joint payoff, given the tasks they were each individually assigned that round.

For a round with scores $V_1$, $V_2$ and each partner's own best-possible score $V_{1,max}$, $V_{2,max}$ (the largest upside across that partner's own task's four design options -- see `analysis/README.md`):

$$E = \frac{V_1 \cdot V_2}{V_{1,max} \cdot V_{2,max}}$$

$E = 1$ when both partners hit their own best case (both pick the same top design and it works out). $E$ is **not** bounded to $[0, 1]$, though -- if one partner's design mismatches their partner's (a coordination failure), that partner's score is the task's downside payoff, which is negative for harder tasks, making $E$ negative too. This is a real feature of the measure, not a bug: it means $E$ captures actual welfare loss from miscoordination, not just "how far from the best outcome."

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

task = pd.read_csv("task_data.csv")
task_summary = pd.read_csv("task_summary.csv")
task.head()

,arm,session,round,username_1,username_2,task_1,task_2,design_1,design_2,strategy_1,strategy_2,collabBelief_1,collabBelief_2,usedRobot_1,usedRobot_2,score_1,score_2
0,control,1,1,user0011,user0012,6,1,M,M,C,C,72,79,False,False,122.0,122.0
1,control,1,2,user0011,user0012,0,13,Y,L,I,C,83,60,False,False,50.0,-85.0
2,control,1,3,user0011,user0012,20,2,Y,L,I,C,50,70,False,False,50.0,-28.0
3,control,1,4,user0011,user0012,15,4,K,K,C,C,100,75,False,False,105.0,100.0
4,control,1,5,user0011,user0012,3,25,L,Y,C,I,0,60,False,False,11.0,50.0


## Drop missing data

Same as `outcome_analysis.ipynb`: a couple of rounds have an undefined strategy on one side, which is also exactly when `score_1`/`score_2` are blank (the server's scoring logic can't compute a score when a design fails to register -- see `results/README.md#task_csv--decision-task-rounds`). Dropping the undefined-strategy rows removes the blank-score rows too.

In [2]:
missing = (task["strategy_1"] == "undefined") | (task["strategy_2"] == "undefined")
print(f"Dropping {missing.sum()} of {len(task)} rounds with an undefined strategy.")
assert (task.loc[missing, "score_1"].isna() & task.loc[missing, "score_2"].isna()).all()

task = task[~missing].copy()
task["pair_id"] = task["username_1"] + "_" + task["username_2"]

Dropping 2 of 780 rounds with an undefined strategy.


## Compute `E` for each round

Each partner's max possible score is the larger of `V_A_CC` (the best collaborative design's upside) and `V_Y_IC` (the individual design's fixed upside) for their own assigned task -- i.e. the best of all four design options' upsides, ignoring what they or their partner actually chose.

In [3]:
real_tasks = task_summary[task_summary["task_difficulty"].notna()].copy()
real_tasks["V_max"] = real_tasks[["V_A_CC", "V_Y_IC"]].max(axis=1)
vmax_by_index = real_tasks.set_index("task_index")["V_max"]

assert task["task_1"].isin(vmax_by_index.index).all()
assert task["task_2"].isin(vmax_by_index.index).all()

v1_max = task["task_1"].map(vmax_by_index)
v2_max = task["task_2"].map(vmax_by_index)
task["E"] = (task["score_1"] * task["score_2"]) / (v1_max * v2_max)

task["E"].describe()

count    778.000000
mean       0.650347
std        0.452372
min       -1.154201
25%        0.205410
50%        0.877049
75%        1.000000
max        1.000000
Name: E, dtype: float64

### Sanity check against the categorical outcome

`E` should track the three `outcome_analysis.ipynb` categories in the expected direction: high for successful collaboration, modest for mutual independence (both get the fixed individual payoff), and negative for coordination failure on harder tasks (the mismatched partner takes the downside).

In [4]:
def classify_outcome(row):
    s1, s2 = row["strategy_1"], row["strategy_2"]
    if s1 == "C" and s2 == "C":
        return "successful collaboration"
    if s1 == "I" and s2 == "I":
        return "mutual independence"
    return "coordination failure"


OUTCOME_ORDER = ["successful collaboration", "mutual independence", "coordination failure"]
task["outcome"] = task.apply(classify_outcome, axis=1)
task.groupby("outcome")["E"].describe().reindex(OUTCOME_ORDER)

,count,mean,std,min,25%,50%,75%,max
outcome,,,,,,,,
successful collaboration,562.0,0.904790,0.124257,0.556981,0.807692,1.000000,1.000000,1.000000
mutual independence,116.0,0.194395,0.015206,0.168691,0.181422,0.192367,0.204918,0.235849
coordination failure,100.0,-0.250721,0.295165,-1.154201,-0.375257,-0.153846,-0.061538,0.088462


## `E` by arm

The numeric counterpart to the outcome-frequency table -- mean, spread, and range of `E`, by arm and overall.

In [5]:
e_by_arm = task.groupby("arm")["E"].agg(["count", "mean", "std", "min", "median", "max"])
e_by_arm.loc["overall"] = task["E"].agg(["count", "mean", "std", "min", "median", "max"])
e_by_arm.round(3)

,count,mean,std,min,median,max
arm,,,,,,
control,358.0,0.563,0.482,-1.154,0.774,1.0
treatment,420.0,0.725,0.412,-1.117,0.882,1.0
overall,778.0,0.650,0.452,-1.154,0.877,1.0


## Inferential statistics: pair-level aggregation

Same non-independence issue as `outcome_analysis.ipynb`: each pair collapsed to its own mean `E`, giving one independent observation per pair (12 control, 14 treatment).

In [6]:
pair_summary = task.groupby(["arm", "pair_id"])["E"].mean().reset_index()

control_vals = pair_summary.loc[pair_summary["arm"] == "control", "E"]
treatment_vals = pair_summary.loc[pair_summary["arm"] == "treatment", "E"]


def cohens_d(a, b):
    n_a, n_b = len(a), len(b)
    pooled_sd = (((n_a - 1) * a.var(ddof=1) + (n_b - 1) * b.var(ddof=1)) / (n_a + n_b - 2)) ** 0.5
    return (a.mean() - b.mean()) / pooled_sd


t_stat, t_p = stats.ttest_ind(treatment_vals, control_vals, equal_var=False)
u_stat, u_p = stats.mannwhitneyu(treatment_vals, control_vals, alternative="two-sided")

print(f"control:   n={len(control_vals)}, mean E={control_vals.mean():.3f}, sd={control_vals.std():.3f}")
print(f"treatment: n={len(treatment_vals)}, mean E={treatment_vals.mean():.3f}, sd={treatment_vals.std():.3f}")
print(f"Cohen's d = {cohens_d(treatment_vals, control_vals):.3f}")
print(f"Welch t = {t_stat:.3f}, p = {t_p:.4f}")
print(f"Mann-Whitney U = {u_stat:.1f}, p = {u_p:.4f}")

control:   n=12, mean E=0.562, sd=0.303
treatment: n=14, mean E=0.725, sd=0.314
Cohen's d = 0.529
Welch t = 1.347, p = 0.1906
Mann-Whitney U = 115.0, p = 0.1165


## Round-level model: linear mixed model with a random intercept per pair

Because `E` is continuous rather than binary, this uses `statsmodels`' `MixedLM` -- a proper REML-estimated linear mixed model, not the variational-Bayes approximation `outcome_analysis.ipynb` had to work around for the binary outcome. Cross-checked against GEE (Gaussian family, cluster-robust SEs on `pair_id`) as before, but this time we'd expect the two to agree closely even for the pair-level `arm` term, since `MixedLM` doesn't share `BinomialBayesMixedGLM.fit_vb()`'s mean-field anti-conservatism.

In [7]:
mixed_basic = smf.mixedlm("E ~ arm", data=task, groups=task["pair_id"]).fit()
print(mixed_basic.summary())

          Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: E        
No. Observations:   778     Method:             REML     
No. Groups:         26      Scale:              0.1135   
Min. group size:    29      Log-Likelihood:     -301.5849
Max. group size:    30      Converged:          Yes      
Mean group size:    29.9                                 
---------------------------------------------------------
                 Coef. Std.Err.   z   P>|z| [0.025 0.975]
---------------------------------------------------------
Intercept        0.562    0.089 6.300 0.000  0.387  0.737
arm[T.treatment] 0.163    0.122 1.343 0.179 -0.075  0.401
Group Var        0.092    0.083                          



In [8]:
gee_basic = smf.gee("E ~ arm", groups="pair_id", data=task, family=sm.families.Gaussian()).fit()
print(gee_basic.summary())

                               GEE Regression Results                              
Dep. Variable:                           E   No. Observations:                  778
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  29
                      Estimating Equations   Max. cluster size:                  30
Family:                           Gaussian   Mean cluster size:                29.9
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 28 Aug 2026   Scale:                           0.198
Covariance type:                    robust   Time:                         22:12:37
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.5628      0.084      6.718      0.000       0.399  

As expected, `MixedLM` and GEE agree closely (coef ~0.16, p ~0.18 in both) -- no repeat of the `outcome_analysis.ipynb` mixed-model-vs-GEE discrepancy, confirming that was specifically a variational-Bayes artifact rather than something intrinsic to mixed models with pair-level covariates. The basic `arm` effect on efficiency is not significant, consistent with the categorical analysis.

## Adding task difficulty

Same derived covariates as `outcome_analysis.ipynb` (`max_difficulty_c`, `diff_difficulty_c`, mean-centered), and the same model shape: `E ~ arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c`.

In [9]:
difficulty_by_index = real_tasks.set_index("task_index")["task_difficulty"]
difficulty_1 = task["task_1"].map(difficulty_by_index).astype(int)
difficulty_2 = task["task_2"].map(difficulty_by_index).astype(int)
task["max_difficulty"] = np.maximum(difficulty_1, difficulty_2)
task["diff_difficulty"] = (difficulty_1 - difficulty_2).abs()
task["max_difficulty_c"] = task["max_difficulty"] - task["max_difficulty"].mean()
task["diff_difficulty_c"] = task["diff_difficulty"] - task["diff_difficulty"].mean()

difficulty_formula = "E ~ arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c"

mixed_difficulty = smf.mixedlm(difficulty_formula, data=task, groups=task["pair_id"]).fit()
print(mixed_difficulty.summary())

                    Mixed Linear Model Regression Results
Model:                    MixedLM        Dependent Variable:        E        
No. Observations:         778            Method:                    REML     
No. Groups:               26             Scale:                     0.1068   
Min. group size:          29             Log-Likelihood:            -287.3880
Max. group size:          30             Converged:                 Yes      
Mean group size:          29.9                                               
-----------------------------------------------------------------------------
                                   Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------------
Intercept                           0.562    0.089  6.297 0.000  0.387  0.736
arm[T.treatment]                    0.164    0.122  1.346 0.178 -0.075  0.402
max_difficulty_c                   -0.066    0.011 -6.081 0.000 -0.087 -0.045
diff_d

In [10]:
gee_difficulty = smf.gee(
    difficulty_formula, groups="pair_id", data=task, family=sm.families.Gaussian(),
).fit()
print(gee_difficulty.summary())

                               GEE Regression Results                              
Dep. Variable:                           E   No. Observations:                  778
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  29
                      Estimating Equations   Max. cluster size:                  30
Family:                           Gaussian   Mean cluster size:                29.9
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 28 Aug 2026   Scale:                           0.192
Covariance type:                    robust   Time:                         22:12:37
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             

## Interpretation

The efficiency-based analysis independently reproduces every conclusion from the categorical `outcome_analysis.ipynb`, using a completely different (continuous, welfare-weighted) outcome measure:

- **No significant uniform `arm` effect** on efficiency (MixedLM and GEE both ~0.16, p ~0.18) -- matches the non-significant `arm` effect on successful-collaboration rate.
- **`max_difficulty_c` is a robust, strongly significant predictor** (p < 0.001 in both models) -- harder tasks reduce efficiency, consistent with (and expected from, since `E` is built from the same downside payoffs) the categorical result.
- **`arm:diff_difficulty_c` is significant** (MixedLM p = 0.008, GEE p = 0.018): treatment's efficiency is more robust to difficulty mismatch between partners than control's -- the same buffering pattern found for successful collaboration, now visible directly in realized payoffs rather than just the win/loss classification.

The pair-level aggregate test (Welch p = 0.19, Mann-Whitney p = 0.12) is weaker than the categorical version's (Mann-Whitney p = 0.064 for successful collaboration specifically) -- plausible, since `E` folds in the *magnitude* of both wins and losses (including the sometimes-large negative values from high-difficulty coordination failures), which adds variance that a simple win/loss rate doesn't have to absorb. The round-level models, which have more power, tell the more consistent story.

## Does pre-existing social closeness matter?

`analysis/survey_data.csv` has each participant's `social_closeness` -- a pre-task demographic item, "how well do you know your selected partner" (1-5, first-time meeting to very close). Averaged across the two partners, this is a **pair-level** covariate, exactly like `arm`: one value per pair, constant across all 30 of their rounds. Two pairs (`user0043`/`user0044`, `user0047`/`user0048`) are dropped here because one member's `social_closeness` was recorded as the literal string `"undefined"` (see `results/README.md#demographics_csv`).

In [11]:
survey = pd.read_csv("survey_data.csv")
closeness_by_user = pd.Series(
    pd.to_numeric(survey["social_closeness"], errors="coerce").values,
    index=survey["username"],
).to_dict()

closeness_1 = task["username_1"].map(closeness_by_user)
closeness_2 = task["username_2"].map(closeness_by_user)
task["avg_social_closeness"] = (closeness_1 + closeness_2) / 2

closeness_data = task.dropna(subset=["avg_social_closeness"]).copy()
closeness_data["avg_social_closeness_c"] = (
    closeness_data["avg_social_closeness"] - closeness_data["avg_social_closeness"].mean()
)

print(f"n={len(closeness_data)} rounds, {closeness_data['pair_id'].nunique()} pairs "
      f"(dropped {task['pair_id'].nunique() - closeness_data['pair_id'].nunique()} pairs lacking closeness data)")
closeness_data.groupby("pair_id")["avg_social_closeness"].first().describe()

n=718 rounds, 24 pairs (dropped 2 pairs lacking closeness data)


count    24.000000
mean      1.979167
std       1.289415
min       1.000000
25%       1.000000
50%       1.000000
75%       3.000000
max       4.500000
Name: avg_social_closeness, dtype: float64

In [12]:
closeness_formula = "E ~ arm + avg_social_closeness_c"

mixed_closeness = smf.mixedlm(closeness_formula, data=closeness_data, groups=closeness_data["pair_id"]).fit()
print(mixed_closeness.summary())

             Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   E        
No. Observations:     718       Method:               REML     
No. Groups:           24        Scale:                0.1050   
Min. group size:      29        Log-Likelihood:       -251.0950
Max. group size:      30        Converged:            Yes      
Mean group size:      29.9                                     
---------------------------------------------------------------
                       Coef. Std.Err.   z   P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept              0.576    0.087 6.615 0.000  0.405  0.746
arm[T.treatment]       0.170    0.120 1.420 0.156 -0.065  0.405
avg_social_closeness_c 0.080    0.047 1.680 0.093 -0.013  0.172
Group Var              0.077    0.078                          



In [13]:
gee_closeness = smf.gee(
    closeness_formula, groups="pair_id", data=closeness_data, family=sm.families.Gaussian(),
).fit()
print(gee_closeness.summary())

                               GEE Regression Results                              
Dep. Variable:                           E   No. Observations:                  718
Model:                                 GEE   No. clusters:                       24
Method:                        Generalized   Min. cluster size:                  29
                      Estimating Equations   Max. cluster size:                  30
Family:                           Gaussian   Mean cluster size:                29.9
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 28 Aug 2026   Scale:                           0.173
Covariance type:                    robust   Time:                         22:12:37
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  0.5767      0.093      6.218      0

### Discussion

`avg_social_closeness` shows a positive coefficient in both models -- pairs who reported knowing each other better before the study tend to have higher efficiency -- but the two models disagree on how confident to be: `MixedLM` gives p ≈ 0.09 (a trend), GEE gives p ≈ 0.02 (significant). `arm` is essentially unchanged from the basic model either way, so this isn't confounding the treatment comparison.

This split is smaller than the dramatic mixed-model-vs-GEE gap in `outcome_analysis.ipynb`, and for a good reason: `MixedLM` uses proper REML estimation, not the variational-Bayes approximation that caused that earlier gap. But `avg_social_closeness` is still a **pair-level** covariate (constant across a pair's 30 rounds, just like `arm`), so its real information content comes from only 24 independent pairs, not 718 rounds -- `MixedLM`'s REML variance structure reflects that more conservatively than GEE's cluster-robust sandwich estimator does here. Read this as a **plausible but unconfirmed** trend, not a settled result -- consistent with how every other pair-level effect in this analysis has needed a matching caveat once the clustering is taken seriously.